# Technical Documentation Assistant (RAG)


In [2]:
!pip install -q langchain-community langchain-chroma langchain-text-splitters chromadb pypdf sentence-transformers transformers torch accelerate

In [3]:
from google.colab import files
uploaded = files.upload()

Saving python_beginners_guide.pdf to python_beginners_guide.pdf


In [4]:
pdf_path = list(uploaded.keys())[0]
print(pdf_path)

python_beginners_guide.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

loader = PyPDFLoader(pdf_path)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=20)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectordb = Chroma.from_documents(documents=chunks, embedding=embeddings)

print('Chunks:', len(chunks))

/tmp/ipykernel_1151/392455385.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_1151/392455385.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks: 109


In [6]:
from transformers import pipeline
llm = pipeline('text-generation', model='TinyLlama/TinyLlama-1.1B-Chat-v1.0')

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [7]:
question = input('Enter Question: ')
retrieved_docs = vectordb.similarity_search(question, k=10)

unique_docs=[]
seen=set()
for doc in retrieved_docs:
    content=doc.page_content.strip()
    if content not in seen:
        seen.add(content)
        unique_docs.append(doc)
    if len(unique_docs)==3:
        break

context='\n\n'.join([d.page_content for d in unique_docs])

prompt=f'''You are a Technical Documentation Assistant. Answer ONLY from the documentation context. If not found, say: I could not find this information in the uploaded documentation.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:'''

result=llm(prompt,max_new_tokens=150,do_sample=False)
print(result[0]['generated_text'])

Enter Question: what is python


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


You are a Technical Documentation Assistant. Answer ONLY from the documentation context. If not found, say: I could not find this information in the uploaded documentation.

Context:
1. Introduction to Python
Python is a general-purpose, high-level programming language created by Guido van Rossum and first
released in 1991. It emphasizes readability, a clean syntax, and a philosophy summarized in the "Zen

of Python": code should be simple, explicit, and easy to understand. Because of this design philosophy,
Python has become one of the most widely used languages in the world, spanning web development,
data analysis, automation, scientific computing, and artificial intelligence.

Python Programming — A Beginner's Guide
Page 7 of 20

Question:
what is python

Answer:
Python is a programming language that is easy to learn and use. It is a high-level language that is
designed to be easy to read and understand. Python is a general-purpose language that can be used for
various applications,

In [8]:
for i,doc in enumerate(unique_docs,1):
    print(f'\nReference {i} (Page {doc.metadata.get("page",0)+1})')
    print(doc.page_content[:700])


Reference 1 (Page 3)
1. Introduction to Python
Python is a general-purpose, high-level programming language created by Guido van Rossum and first
released in 1991. It emphasizes readability, a clean syntax, and a philosophy summarized in the "Zen

Reference 2 (Page 3)
of Python": code should be simple, explicit, and easy to understand. Because of this design philosophy,
Python has become one of the most widely used languages in the world, spanning web development,
data analysis, automation, scientific computing, and artificial intelligence.

Reference 3 (Page 7)
Python Programming — A Beginner's Guide
Page 7 of 20
